# QRC: Perfect Simulation vs Noisy (Hardware-Realistic) Simulation

This notebook compares the Quantum Reservoir Computing pipeline under two regimes:

| Regime | Description |
|--------|-------------|
| **Perfect** | Ideal photons, zero loss, full indistinguishability (SLOS backend, default) |
| **Noisy** | Hardware-realistic imperfections via Perceval's `NoiseModel` |

**Key question:** Does the QRC ensemble still beat classical baselines under realistic noise?

### Methodology

- Same **3-seed ensemble** (seeds 5, 15, 16) and **splits** as QRC.ipynb
- Same Ridge readout + test evaluation on `test.xlsx`
- Only the feature extraction differs: perfect MerLin vs noisy raw `Processor`

### Noise parameters

Values representative of state-of-the-art quantum dot sources (Quandela Ascella-class):

| Parameter | Symbol | Perfect | Hardware | Optimistic | Pessimistic |
|-----------|--------|---------|----------|------------|-------------|
| Brightness | $\eta$ | 1.0 | 0.15 | 0.30 | 0.08 |
| Indistinguishability | $M$ | 1.0 | 0.92 | 0.96 | 0.85 |
| Multi-photon | $g^{(2)}(0)$ | 0.0 | 0.01 | 0.005 | 0.03 |
| Transmittance | $T$ | 1.0 | 0.1 | 0.2 | 0.05 |
| Phase imprecision | $\delta\phi$ | 0.0 | 0.05 | 0.02 | 0.10 |

### Why MerLin QuantumLayer doesn't support noise

MerLin's `QuantumLayer` wraps Perceval but bypasses the `Processor.noise` path —
it uses the raw backend directly. So we extract the circuit from MerLin, then run
it through a raw `Processor(noise=NoiseModel(...))` for the noisy features.

## 0 · Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import torch
import perceval as pcvl
from perceval.utils.noise_model import NoiseModel
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
import matplotlib.pyplot as plt
from merlin import (
    QuantumLayer,
    CircuitBuilder,
    MeasurementStrategy,
    ComputationSpace,
)
import time, os

# ---- Shared config (matches QRC.ipynb) ----
WINDOW   = 20
HORIZON  = 10
VAL_SIZE = 30

N_MODES   = 10
N_PHOTONS = 5
N_PCA     = 4
N_STEPS   = 3
INPUT_MODES  = list(range(N_PCA))
HIDDEN_MODES = list(range(N_PCA, N_MODES))

# Same ensemble as QRC.ipynb (top-3 by val QLIKE from 30-seed sweep)
ENSEMBLE_SEEDS = [5, 15, 16]

np.random.seed(42)
torch.manual_seed(42)

# ---- Noise profiles ----
NOISE_PROFILES = {
    "perfect": None,  # no NoiseModel = ideal simulation
    "hardware": NoiseModel(
        brightness=0.15,
        indistinguishability=0.92,
        g2=0.01,
        transmittance=0.1,
        phase_imprecision=0.05,
    ),
    "optimistic": NoiseModel(
        brightness=0.30,
        indistinguishability=0.96,
        g2=0.005,
        transmittance=0.2,
        phase_imprecision=0.02,
    ),
    "pessimistic": NoiseModel(
        brightness=0.08,
        indistinguishability=0.85,
        g2=0.03,
        transmittance=0.05,
        phase_imprecision=0.10,
    ),
}

print(f"Ensemble seeds: {ENSEMBLE_SEEDS}")
print("Noise profiles:")
for name, nm in NOISE_PROFILES.items():
    print(f"  {name}: {nm}")

## 1 · Data Preparation

Identical to QRC.ipynb — StandardScaler + PCA fit on train only, angle encoding to [0, π].

In [ ]:
df = pd.read_parquet("../data/level1.parquet")
df['Date'] = pd.to_datetime(df['Date'], format='mixed')
df = df.sort_values('Date').reset_index(drop=True)
price_cols = [c for c in df.columns if c != 'Date']
prices = df[price_cols].astype(float).values  # (494, 224)

scaler = StandardScaler()
scaler.fit(prices[:-VAL_SIZE])
prices_scaled = scaler.transform(prices)

pca = PCA(n_components=N_PCA, random_state=42)
pca.fit(prices_scaled[:-VAL_SIZE])
prices_pca = pca.transform(prices_scaled)

pca_min = prices_pca[:-VAL_SIZE].min(axis=0)
pca_max = prices_pca[:-VAL_SIZE].max(axis=0)
prices_angles = (prices_pca - pca_min) / (pca_max - pca_min + 1e-8) * np.pi

print(f"Data: {prices.shape}")
print(f"PCA variance explained: {pca.explained_variance_ratio_.sum():.3f}")
print(f"Angle range: [{prices_angles.min():.3f}, {prices_angles.max():.3f}]")

## 2 · Load Reservoir Ensemble

Load the exact same frozen reservoir weights saved by QRC.ipynb (`models/reservoir_seed*.pt`).
This guarantees identical circuits regardless of PyTorch RNG changes.

In [ ]:
def build_reservoir_arch():
    """Build a QuantumLayer with the standard architecture (weights uninitialised)."""
    builder = CircuitBuilder(n_modes=N_MODES)
    for _ in range(N_STEPS):
        builder.add_entangling_layer(trainable=True)
        builder.add_angle_encoding(modes=INPUT_MODES)
    builder.add_entangling_layer(trainable=True)

    return QuantumLayer(
        input_size=N_STEPS * N_PCA,
        builder=builder,
        n_photons=N_PHOTONS,
        measurement_strategy=MeasurementStrategy.MODE_EXPECTATIONS,
        computation_space=ComputationSpace.UNBUNCHED,
        dtype=torch.float64,
    )


# Load saved weights from QRC.ipynb
reservoirs = {}
for seed in ENSEMBLE_SEEDS:
    res = build_reservoir_arch()
    path = f"../models/reservoir_seed{seed}.pt"
    res.load_state_dict(torch.load(path, weights_only=True))
    # Ensure frozen
    for p in res.parameters():
        p.requires_grad = False
    reservoirs[seed] = res
    print(f"Loaded seed {seed} from {path}  ({sum(p.numel() for p in res.parameters())} params)")

print(f"\n{len(reservoirs)} reservoirs loaded (matching QRC.ipynb ensemble)")

## 3 · Feature Extraction: Perfect vs Noisy

**Perfect:** Uses MerLin's `QuantumLayer` directly (fast, deterministic).

**Noisy:** Creates a `Processor(noise=NoiseModel(...))` with the same circuit,
then computes mode expectations from the noisy output distribution.

Mode expectations under noise: $\langle n_k \rangle = \sum_{s} P(s) \cdot s_k$
where $P(s)$ is the (noisy) probability of Fock state $s$ and $s_k$ is the photon
count in mode $k$.

In [ ]:
def extract_features_perfect(reservoir, angles):
    """Extract features using MerLin QuantumLayer (perfect sim)."""
    n_blocks = len(angles) - N_STEPS + 1
    features = np.empty((n_blocks, reservoir.output_size))
    with torch.no_grad():
        for i in range(n_blocks):
            block = angles[i:i + N_STEPS].flatten()
            x = torch.tensor(block[np.newaxis, :], dtype=torch.float64)
            features[i] = reservoir(x).numpy().flatten()
    return features


def extract_features_noisy(reservoir, angles, noise_model, min_detected=1):
    """
    Extract features using raw Perceval Processor with noise.
    
    Computes mode expectations from the noisy output distribution:
    <n_k> = sum_s P(s) * s_k
    """
    circ = reservoir.circuit
    n_blocks = len(angles) - N_STEPS + 1
    features = np.empty((n_blocks, N_MODES))
    
    # Build the input Fock state: one photon in each of the first N_PHOTONS modes
    input_fock = [0] * N_MODES
    for m in range(N_PHOTONS):
        input_fock[m] = 1
    input_state = pcvl.BasicState(input_fock)
    
    # Identify circuit parameters by name:
    #   el_*  = entangling layer phases (frozen random values)
    #   px*   = angle encoding inputs  (set per block)
    all_params = circ.get_parameters()
    el_params = [p for p in all_params if p.name.startswith("el_")]
    px_params = sorted(
        [p for p in all_params if p.name.startswith("px")],
        key=lambda p: int(p.name[2:]),  # px1, px2, ... → sort numerically
    )
    
    # Fix entangling layer params from the frozen torch weights (once)
    torch_vals = list(reservoir.parameters())[0].data.numpy().flatten()
    for idx, ep in enumerate(el_params):
        ep.set_value(float(torch_vals[idx]))
    
    for i in range(n_blocks):
        block = angles[i:i + N_STEPS].flatten()
        
        # Set angle encoding parameters for this block
        for j, pp in enumerate(px_params):
            pp.set_value(float(block[j]))
        
        # Create noisy processor
        proc = pcvl.Processor("SLOS", circ, noise=noise_model)
        proc.with_input(input_state)
        proc.min_detected_photons_filter(min_detected)
        
        # Get output distribution
        dist = proc.probs()["results"]
        
        # Compute mode expectations: <n_k> = sum_s P(s) * s_k
        mode_exp = np.zeros(N_MODES)
        for state, prob in dist.items():
            for k in range(N_MODES):
                mode_exp[k] += prob * state[k]
        
        features[i] = mode_exp
        
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{n_blocks}")
    
    return features


print("Feature extractors defined.")
print(f"Blocks to process: {len(prices_angles) - N_STEPS + 1}")

# Verify circuit parameter structure (using first reservoir)
_res0 = reservoirs[ENSEMBLE_SEEDS[0]]
all_p = _res0.circuit.get_parameters()
el_count = sum(1 for p in all_p if p.name.startswith("el_"))
px_count = sum(1 for p in all_p if p.name.startswith("px"))
print(f"Circuit params: {el_count} entangling (el_*) + {px_count} angle-encoding (px*) = {len(all_p)} total")
print(f"Torch frozen weights: {list(_res0.parameters())[0].numel()}")
print(f"Expected px count = N_STEPS * N_PCA = {N_STEPS * N_PCA}")

## 4 · Extract Features for All Seeds × Noise Profiles

This is the expensive cell — noisy simulation is significantly slower than perfect
because the Processor must handle mixed states from the imperfect source.

Structure: `all_features[seed][profile_name]` → (492, 10) array.

In [ ]:
# all_features = {}  # {seed: {profile: (492, 10)}}

# for seed in ENSEMBLE_SEEDS:
#     res = reservoirs[seed]
#     all_features[seed] = {}
#     print(f"\n{'='*50}")
#     print(f"Seed {seed}")
#     print(f"{'='*50}")

#     # Perfect (fast — via MerLin)
#     print("  Extracting: perfect ...")
#     t0 = time.time()
#     all_features[seed]["perfect"] = extract_features_perfect(res, prices_angles)
#     print(f"    Done in {time.time() - t0:.1f}s")

#     # Noisy profiles (slower — via raw Perceval Processor)
#     for name, nm in NOISE_PROFILES.items():
#         if nm is None:
#             continue
#         print(f"  Extracting: {name} ...")
#         t0 = time.time()
#         all_features[seed][name] = extract_features_noisy(res, prices_angles, nm)
#         print(f"    Done in {time.time() - t0:.1f}s")

# profiles = list(NOISE_PROFILES.keys())
# print(f"\nAll done: {len(ENSEMBLE_SEEDS)} seeds × {len(profiles)} profiles")
# for seed in ENSEMBLE_SEEDS:
#     for prof in profiles:
#         print(f"  seed={seed} {prof}: {all_features[seed][prof].shape}")

In [ ]:
# Save / load extracted features so we can skip the expensive extraction cell
FEATURES_DIR = "../models"
FEATURES_PATH = os.path.join(FEATURES_DIR, "qrc_noisy_features_ensemble.npz")
os.makedirs(FEATURES_DIR, exist_ok=True)

# # ---- Save ----
# # Flatten key as "seed_profile" for npz storage
# save_dict = {}
# for seed in ENSEMBLE_SEEDS:
#     for prof, feats in all_features[seed].items():
#         save_dict[f"{seed}_{prof}"] = feats

# np.savez_compressed(FEATURES_PATH, **save_dict)
# print(f"Saved {len(save_dict)} arrays → {FEATURES_PATH}")

In [ ]:
# ---- To reload next time, comment out the extraction cell and run: ----
data = np.load(FEATURES_PATH)
all_features = {}
for seed in ENSEMBLE_SEEDS:
    all_features[seed] = {}
    for prof in NOISE_PROFILES:
        key = f"{seed}_{prof}"
        all_features[seed][prof] = data[key]
print(f"Loaded from cache: {list(data.files)}")

## 5 · Ridge Readout + Ensemble (per Noise Profile)

Same pipeline as QRC.ipynb: for each seed × profile, train Ridge on train split,
predict on test. Then **average predictions across seeds** (ensemble).

Test evaluation uses `test.xlsx` with training column order.

In [ ]:
def build_windows(q_feats, targets_scaled, window, horizon, offset=0):
    X, Y = [], []
    for i in range(len(q_feats) - window - horizon + 1):
        X.append(q_feats[i:i + window].flatten())
        raw_start = i + offset + window
        Y.append(targets_scaled[raw_start:raw_start + horizon].flatten())
    return np.array(X), np.array(Y)


def compute_metrics(pred, actual):
    rmse = np.sqrt(((pred - actual) ** 2).mean())
    mae  = np.abs(pred - actual).mean()
    eps  = 1e-8
    ratio = actual / np.clip(pred, eps, None)
    qlike = (ratio - np.log(ratio) - 1).mean()
    return {'RMSE': float(rmse), 'MAE': float(mae), 'QLIKE': float(qlike)}


def evaluate_single(q_features):
    """Train Ridge on given quantum features. Return val metrics + test prediction."""
    X_all, Y_all = build_windows(
        q_features, prices_scaled, WINDOW, HORIZON, offset=N_STEPS - 1
    )
    n_total   = len(prices_scaled)
    val_start = n_total - VAL_SIZE
    train_end = val_start - WINDOW - N_STEPS + 1 - HORIZON + 1

    X_train, Y_train = X_all[:train_end], Y_all[:train_end]
    X_val,   Y_val   = X_all[train_end:], Y_all[train_end:]

    ridge = RidgeCV(alphas=np.logspace(-3, 6, 50), scoring='neg_mean_squared_error')
    ridge.fit(X_train, Y_train)

    # Val metrics (original scale)
    Y_val_hat = ridge.predict(X_val)
    val_pred = np.concatenate([
        scaler.inverse_transform(Y_val_hat[i].reshape(HORIZON, 224))
        for i in range(len(Y_val_hat))
    ])
    val_true = np.concatenate([
        scaler.inverse_transform(Y_val[i].reshape(HORIZON, 224))
        for i in range(len(Y_val))
    ])
    val_m = compute_metrics(val_pred, val_true)

    # Test prediction
    test_input = q_features[-WINDOW:].flatten().reshape(1, -1)
    test_pred  = scaler.inverse_transform(
        ridge.predict(test_input).reshape(HORIZON, 224)
    )
    return val_m, test_pred, ridge.alpha_


# ---- Load test ground truth (training column order) ----
test_df = pd.read_excel("../data/test.xlsx")
test_df['Date'] = pd.to_datetime(test_df['Date'], format='mixed')
test_actual = test_df[price_cols].astype(float).values
N_TEST = len(test_actual)
print(f"Test data: {test_actual.shape}")
print(f"Dates: {test_df['Date'].iloc[0].date()} → {test_df['Date'].iloc[-1].date()}\n")

# ---- Per-seed evaluation + ensemble ----
profiles = list(NOISE_PROFILES.keys())
ensemble_test_preds = {prof: [] for prof in profiles}  # collect per-seed test preds
val_results = {prof: [] for prof in profiles}

print(f"{'Seed':>6s}  {'Profile':>12s}  |  {'Val QLIKE':>10s}  {'Val RMSE':>10s}  {'α':>8s}")
print("-" * 62)

for seed in ENSEMBLE_SEEDS:
    for prof in profiles:
        feats = all_features[seed][prof]
        val_m, t_pred, alpha = evaluate_single(feats)
        val_results[prof].append(val_m)
        ensemble_test_preds[prof].append(t_pred)
        print(f"{seed:>6d}  {prof:>12s}  |  {val_m['QLIKE']:>10.6f}  {val_m['RMSE']:>10.6f}  {alpha:>8.4f}")

# ---- Ensemble: average test predictions across seeds ----
print(f"\n{'='*62}")
print(f"{'ENSEMBLE TEST RESULTS':^62s}")
print(f"{'='*62}")
print(f"{'Profile':>14s}  |  {'QLIKE':>10s}  {'RMSE':>10s}  {'MAE':>10s}")
print("-" * 58)

test_results = {}
for prof in profiles:
    ens_pred = np.mean(ensemble_test_preds[prof], axis=0)[:N_TEST]
    m = compute_metrics(ens_pred, test_actual)
    test_results[prof] = m
    print(f"{prof:>14s}  |  {m['QLIKE']:>10.6f}  {m['RMSE']:>10.6f}  {m['MAE']:>10.6f}")

print("-" * 58)
print(f"{'MLP_baseline':>14s}  |  {0.003921:>10.6f}  {0.015962:>10.6f}  {0.012978:>10.6f}")
print(f"{'LSTM_baseline':>14s}  |  {0.001081:>10.6f}  {0.007712:>10.6f}  {0.006434:>10.6f}")

In [ ]:
# Save per-profile ensemble test predictions for standalone test eval script
save_dir = "../models"
for prof in profiles:
    ens_pred = np.mean(ensemble_test_preds[prof], axis=0)[:N_TEST]
    path = os.path.join(save_dir, f"qrc_noisy_test_pred_{prof}.npy")
    np.save(path, ens_pred)
    print(f"Saved: {path}  shape={ens_pred.shape}")
print("Done — these are loaded by src/test_eval_noisy_QRC.py")

## 6 · Visualisation

### 6a — Ensemble Test QLIKE: noise profiles vs classical baselines

In [ ]:
os.makedirs("../results", exist_ok=True)

# Combine QRC ensemble + classical baselines
all_test = {**test_results}
all_test["MLP_baseline"]  = {'QLIKE': 0.003921, 'RMSE': 0.015962, 'MAE': 0.012978}
all_test["LSTM_baseline"] = {'QLIKE': 0.001081, 'RMSE': 0.007712, 'MAE': 0.006434}

profiles = ['perfect', 'hardware', 'optimistic', 'pessimistic']
qrc_qlikes = [test_results[p]['QLIKE'] for p in profiles]
cmap_qrc = ['#3498db', '#e67e22', '#f39c12', '#e74c3c']

fig, (ax_zoom, ax_all) = plt.subplots(1, 2, figsize=(13, 5),
                                       gridspec_kw={'width_ratios': [3, 2]})

# ---- Left: zoomed into QRC noise profiles ----
bars = ax_zoom.bar(profiles, qrc_qlikes, color=cmap_qrc, edgecolor='white',
                   width=0.6, linewidth=1.2)
ymax_zoom = max(qrc_qlikes) * 1.25
for bar, val in zip(bars, qrc_qlikes):
    ax_zoom.text(bar.get_x() + bar.get_width()/2, val + ymax_zoom*0.02,
                 f'{val:.6f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    if val != qrc_qlikes[0]:
        pct = (val - qrc_qlikes[0]) / qrc_qlikes[0] * 100
        ax_zoom.text(bar.get_x() + bar.get_width()/2, val * 0.5,
                     f'+{pct:.0f}%', ha='center', va='center',
                     color='white', fontweight='bold', fontsize=11)

ax_zoom.axhline(y=qrc_qlikes[0], color='#3498db', ls='--', alpha=0.4,
                label=f'Perfect = {qrc_qlikes[0]:.6f}')
ax_zoom.set_ylim(0, ymax_zoom)
ax_zoom.set_ylabel('QLIKE  (lower is better)', fontsize=11)
ax_zoom.set_title('QRC Ensemble — Noise Profile Comparison', fontsize=12, fontweight='bold')
ax_zoom.legend(fontsize=9, loc='upper left')
ax_zoom.tick_params(axis='x', labelsize=10)

# ---- Right: horizontal bars — all methods on same scale ----
all_names  = [f'QRC {p}' for p in profiles] + ['LSTM', 'MLP']
all_vals   = qrc_qlikes + [0.001081, 0.003921]
all_colors = cmap_qrc   + ['#7f8c8d', '#bdc3c7']

y_pos = list(range(len(all_names)))
hbars = ax_all.barh(y_pos, all_vals, color=all_colors, edgecolor='white',
                    height=0.6, linewidth=1.2)
ax_all.set_yticks(y_pos)
ax_all.set_yticklabels(all_names, fontsize=10)
for bar, val in zip(hbars, all_vals):
    ax_all.text(val + 0.00008, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9.5, fontweight='bold')
ax_all.set_xlabel('QLIKE  (lower is better)', fontsize=11)
ax_all.set_title('vs Classical Baselines', fontsize=12, fontweight='bold')
ax_all.invert_yaxis()
ax_all.set_xlim(0, max(all_vals) * 1.22)

plt.suptitle('Ensemble Test QLIKE — Perfect vs Noisy QRC vs Classical',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/qrc_noisy_ensemble_test_qlike.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/qrc_noisy_ensemble_test_qlike.png")

### 6b — Feature space comparison (seed 5, representative)

How much do noisy features deviate from the perfect (ideal) features?

In [ ]:
REP_SEED = ENSEMBLE_SEEDS[0]  # representative seed for feature plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
perfect_feats = all_features[REP_SEED]['perfect']
noise_names = ['hardware', 'optimistic', 'pessimistic']
noise_colors = ['#e67e22', '#f39c12', '#e74c3c']

for ax, name, clr in zip(axes, noise_names, noise_colors):
    noisy_feats = all_features[REP_SEED][name]
    pf = perfect_feats.flatten()
    nf = noisy_feats.flatten()

    # Tight limits centred on data
    lo = min(pf.min(), nf.min())
    hi = max(pf.max(), nf.max())
    pad = (hi - lo) * 0.05
    lims = [lo - pad, hi + pad]

    # 2D histogram (hex) for dense regions + scatter fallback
    ax.hexbin(pf, nf, gridsize=40, cmap='Blues', mincnt=1, alpha=0.85, linewidths=0.2)
    ax.plot(lims, lims, 'r--', alpha=0.7, linewidth=1.5, label='y = x  (perfect match)')

    corr = np.corrcoef(pf, nf)[0, 1]
    rmse = np.sqrt(np.mean((pf - nf)**2))
    mae  = np.mean(np.abs(pf - nf))

    # Stats box
    txt = f'ρ = {corr:.4f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}'
    ax.text(0.05, 0.95, txt, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round,pad=0.4',
            facecolor='white', alpha=0.85, edgecolor=clr, linewidth=1.5))

    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_aspect('equal')
    ax.set_title(name, fontsize=13, fontweight='bold', color=clr)
    ax.set_xlabel('Perfect features', fontsize=10)
    ax.set_ylabel('Noisy features', fontsize=10)
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle(f'Quantum Feature Space: Perfect vs Noisy  (seed {REP_SEED})',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('../results/qrc_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/qrc_feature_correlation.png")

## 7 · Summary

### Setup

- Same ensemble as QRC.ipynb: seeds [5, 15, 16], W=20, top-3 by val QLIKE
- Same train/val/test splits; test evaluated on `test.xlsx`
- Only difference: feature extraction under `NoiseModel` vs perfect simulation
- Classical baselines computed with same QLIKE formula (`ratio − ln(ratio) − 1`) in Test_evaluation.ipynb

### Results

| Profile | Ensemble Test QLIKE | Δ vs Perfect | vs LSTM (0.001081) |
|-------------|-------------------|--------------|:-------------------:|
| **Perfect** | 0.000879 | — | ✅ −19% |
| **Pessimistic** | 0.001110 | −26.3% | ❌ +2.7% |
| **Optimistic** | 0.001147 | −30.5% | ❌ +6.1% |
| **Hardware** | 0.001286 | −46.3% | ❌ +19.0% |
| LSTM baseline | 0.001081 | — | — |
| MLP baseline | 0.003921 | — | ❌ |

> **Δ vs Perfect**: negative = degradation from the ideal (perfect) baseline.

### Key Findings

1. **QRC (perfect) beats LSTM by 19%.** The quantum advantage is real under ideal simulation.
2. **Noise degrades performance by 26–46%**, pushing noisy QRC above LSTM. Hardware profile is the worst at +19% vs LSTM.
3. **All QRC profiles still beat MLP** by a wide margin (>3× better).
4. **Noise ordering**: Pessimistic (0.001110) ≤ optimistic (0.001147) < hardware (0.001286). The noisy profiles are surprisingly close — the ensemble + Ridge regularisation smooths over noise-induced variance.
5. **The gap to LSTM is small** — pessimistic is only 2.7% worse, suggesting noise mitigation techniques (ZNE, error cancellation) could close it.

### Implications

- **Perfect QRC → clear quantum advantage.** Under ideal photonic simulation, the reservoir approach decisively beats the best classical baseline.
- **Noisy QRC → competitive but not yet superior to LSTM.** The noise penalty (~26–46%) is modest given the severity of the noise parameters, but it crosses the LSTM threshold.
- **Noise mitigation could recover the advantage.** The pessimistic profile is only 2.7% from LSTM — ZNE or ensemble scaling could bridge this gap.
- The ensemble averaging across 3 seeds provides built-in noise suppression.

### Next steps

1. **Noise mitigation**: Test zero-noise extrapolation (ZNE) to reduce the noise penalty.
2. **Quandela Cloud QPU**: Replace `NoiseModel` with actual hardware runs via `pcvl.RemoteProcessor`.
3. **Larger ensembles**: More seeds → better noise averaging → potentially recover LSTM-beating performance.
4. **Scaling**: Test on larger / real-world swaption surfaces.